In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)


In [18]:
data = pd.read_csv("diabetic_data.csv")

print("Dataset loaded successfully!")



Dataset loaded successfully!


In [19]:
print("\nDataset shape:")
print(data.shape)

print("\nFirst 5 rows:")
print(data.head())




Dataset shape:
(101766, 50)

First 5 rows:
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   
3        500364     82442376        Caucasian    Male  [30-40)      ?   
4         16680     42519267        Caucasian    Male  [40-50)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  

In [20]:
print("\nColumn names:")
print(data.columns.tolist())



Column names:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [21]:
print("\nMissing values:")
print(data.isnull().sum())



Missing values:
encounter_id                    0
patient_nbr                     0
race                            0
gender                          0
age                             0
weight                          0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                          0
diag_2                          0
diag_3                          0
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                    

In [22]:
data["readmitted_30d"] = (
    data["readmitted"] == "<30"
).astype(int)


print("\nTarget distribution:")
print(data["readmitted_30d"].value_counts())

print("\nTarget percentages:")
print(
    data["readmitted_30d"]
    .value_counts(normalize=True) * 100
)



Target distribution:
readmitted_30d
0    90409
1    11357
Name: count, dtype: int64

Target percentages:
readmitted_30d
0    88.840084
1    11.159916
Name: proportion, dtype: float64


In [23]:
age_mapping = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
}

data["age_numeric"] = data["age"].map(age_mapping)


In [24]:
data["diag_1"] = data["diag_1"].astype(str)

data["diagnosis_group"] = data["diag_1"].str[:3]


In [25]:
data = data.replace("?", np.nan)

In [26]:
features = [
    "age_numeric",
    "diagnosis_group",
    "time_in_hospital",
    "num_lab_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

X = data[features]

y = data["readmitted_30d"]


In [27]:
print("\nSelected features:")
print(X.head())



Selected features:
   age_numeric diagnosis_group  time_in_hospital  num_lab_procedures  \
0            5             250                 1                  41   
1           15             276                 3                  59   
2           25             648                 2                  11   
3           35               8                 2                  44   
4           45             197                 1                  51   

   num_medications  number_outpatient  number_emergency  number_inpatient  \
0                1                  0                 0                 0   
1               18                  0                 0                 0   
2               13                  2                 0                 1   
3               16                  0                 0                 0   
4                8                  0                 0                 0   

   number_diagnoses  
0                 1  
1                 9  
2                 

In [28]:
numeric_features = [
    "age_numeric",
    "time_in_hospital",
    "num_lab_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

categorical_features = [
    "diagnosis_group"
]


In [29]:

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [30]:

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])



In [31]:
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])




In [32]:
model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    random_state=42
)


In [33]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])


# ============================================================
# 12. SPLIT DATA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))





Training samples: 81412
Testing samples: 20354


In [34]:
print("\nTraining Logistic Regression...")

pipeline.fit(
    X_train,
    y_train
)

print("Model trained successfully!")



Training Logistic Regression...
Model trained successfully!


In [35]:
y_pred = pipeline.predict(X_test)


# Probability that patient will be readmitted
y_probability = pipeline.predict_proba(X_test)[:, 1]


In [36]:
roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("\n========================================")
print("ROC-AUC SCORE")
print("========================================")

print("ROC-AUC:", round(roc_auc, 4))



ROC-AUC SCORE
ROC-AUC: 0.6423


In [37]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n========================================")
print("CONFUSION MATRIX")
print("========================================")

print(cm)




CONFUSION MATRIX
[[18046    37]
 [ 2235    36]]


In [38]:
tn, fp, fn, tp = cm.ravel()


print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)




True Negatives : 18046
False Positives: 37
False Negatives: 2235
True Positives : 36


In [42]:
print("\n========================================")
print("CLASSIFICATION REPORT")
print("========================================")

print(
    classification_report(
        y_test,
        y_pred
    )
)




CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.49      0.02      0.03      2271

    accuracy                           0.89     20354
   macro avg       0.69      0.51      0.49     20354
weighted avg       0.85      0.89      0.84     20354



In [43]:
print("\n========================================")
print("FINAL RESULTS")
print("========================================")

print("ROC-AUC Score   :", round(roc_auc, 4))
print("True Negatives  :", tn)
print("False Positives :", fp)
print("False Negatives :", fn)
print("True Positives  :", tp)




FINAL RESULTS
ROC-AUC Score   : 0.6423
True Negatives  : 18046
False Positives : 37
False Negatives : 2235
True Positives  : 36
